# Indonesian Tweet Emotion Classification with Word2Vec

A text processing coursework experiment with five emotion labels: anger, fear, happy, love, and sadness. It compares mean pretrained Word2Vec vectors + Random Forest with TF-IDF + Word2Vec features + Multinomial Naive Bayes. The CSV, optional slang dictionary, and pretrained vectors are not included. See README for setup.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, ConfusionMatrixDisplay


## 1. Load the data and pretrained vectors

Edit paths below if you use Google Colab or another dataset location. `bahasa_gaul.csv` is optional. The original experiment used a 400-dimensional pretrained Word2Vec text file whose source and license are not documented in the notebook.


In [ ]:
DATA_CSV = Path("data/Twitter_Emotion_Dataset.csv")
WORD2VEC_TXT = Path("data/Word2Vec_400dim.txt")
SLANG_CSV = Path("data/bahasa_gaul.csv")
RANDOM_STATE = 42
for path in (DATA_CSV, WORD2VEC_TXT):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. See README for required local files.")

df = pd.read_csv(DATA_CSV)
required = {"tweet", "label"}
if not required.issubset(df.columns):
    raise ValueError(f"Dataset needs columns {sorted(required)}; found {df.columns.tolist()}")
df = df.dropna(subset=["tweet", "label"]).copy()
print("Rows:", len(df))
print(df["label"].value_counts())
train_df, val_df = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", len(train_df), "Validation:", len(val_df))
vectors = KeyedVectors.load_word2vec_format(str(WORD2VEC_TXT), binary=False)
print("Embedding dimensions:", vectors.vector_size)


## 2. Clean and tokenize tweets

The original notebook removed non-ASCII letters, mentions, placeholders, and URLs. We keep that basic cleaning and split the resulting normalized text into word tokens **before** Word2Vec lookup. The optional two-column slang dictionary maps `slang` to `formal`.


In [ ]:
slang_map = {}
if SLANG_CSV.is_file():
    slang = pd.read_csv(SLANG_CSV)
    if not {"slang", "formal"}.issubset(slang.columns):
        raise ValueError("Slang CSV needs columns 'slang' and 'formal'.")
    slang_map = dict(zip(slang["slang"].astype(str).str.lower(),
                         slang["formal"].astype(str).str.lower()))
    print("Slang entries:", len(slang_map))
else:
    print("Optional slang dictionary missing; continuing without slang normalization.")

def preprocess(text):
    text = re.sub(r"\[USERNAME\]|\[URL\]|https?://\S+|www\.\S+", " ", str(text), flags=re.I)
    text = re.sub(r"@\w+|#\w+", " ", text)
    text = re.sub(r"[^a-zA-Z]+", " ", text).lower()
    tokens = text.split()
    return " ".join(slang_map.get(token, token) for token in tokens).split()

train_tokens = train_df["tweet"].map(preprocess)
val_tokens = val_df["tweet"].map(preprocess)
train_text = train_tokens.map(" ".join)
val_text = val_tokens.map(" ".join)
y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()


## 3. Mean Word2Vec vectors + Random Forest

Unknown words are skipped. Tweets with no known tokens receive a zero vector. The original notebook mistakenly iterated over the normalized string (characters); this version uses word tokens.


In [ ]:
def mean_embedding(tokens, model):
    known = [model[token] for token in tokens if token in model.key_to_index]
    return np.mean(known, axis=0) if known else np.zeros(model.vector_size)

X_train_w2v = np.vstack(train_tokens.map(lambda x: mean_embedding(x, vectors)))
X_val_w2v = np.vstack(val_tokens.map(lambda x: mean_embedding(x, vectors)))
print("Word2Vec matrix shapes:", X_train_w2v.shape, X_val_w2v.shape)
rf_model = RandomForestClassifier(max_depth=10, random_state=RANDOM_STATE)
rf_model.fit(X_train_w2v, y_train)
rf_predictions = rf_model.predict(X_val_w2v)
labels = sorted(df["label"].unique())
print(classification_report(y_val, rf_predictions, labels=labels, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_val, rf_predictions, labels=labels,
                                        xticks_rotation=45, cmap="Blues")
plt.title("Word2Vec + Random Forest")
plt.tight_layout()
plt.show()


## 4. TF-IDF + Word2Vec + Multinomial Naive Bayes

Fit TF-IDF and the embedding scaler on training data only. Dense concatenation may require substantial memory for large vocabularies; this dataset is small. This is an exploratory comparison, not a controlled benchmark.


In [ ]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(train_text)
X_val_tfidf = tfidf.transform(val_text)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_w2v)
X_val_scaled = scaler.transform(X_val_w2v)
X_train_combined = np.hstack([X_train_tfidf.toarray(), X_train_scaled])
X_val_combined = np.hstack([X_val_tfidf.toarray(), X_val_scaled])
nb_model = MultinomialNB()
nb_model.fit(X_train_combined, y_train)
nb_predictions = nb_model.predict(X_val_combined)
print(classification_report(y_val, nb_predictions, labels=labels, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_val, nb_predictions, labels=labels,
                                        xticks_rotation=45, cmap="Blues")
plt.title("TF-IDF + Word2Vec + Multinomial NB")
plt.tight_layout()
plt.show()


## Limitations

The notebook uses one random train/validation split, no independent test set, and an externally supplied Word2Vec file. The original output reported about 35% validation accuracy for Word2Vec + Random Forest and about 36% for the combined Naive Bayes model; because the string/token bug was fixed here, these historical metrics do not describe a rerun of this cleaned notebook. Dataset and embeddings provenance must be verified before sharing them.
